In [ ]:
# Pinned opentelemetry versions prevent chromadb runtime locking conflicts
!pip install -q \
    pypdf pdfplumber \
    langchain langchain-community langchain-huggingface langchain-chroma \
    langchain-text-splitters \
    chromadb \
    sentence-transformers \
    rank_bm25 \
    google-genai \
    tqdm \
    opentelemetry-api==1.22.0 opentelemetry-sdk==1.22.0

print("✅ All packages installed successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.6/105.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 132.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1

In [ ]:
import os
import re
import json
import pickle
import shutil
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from tqdm import tqdm
import pdfplumber
from sentence_transformers import CrossEncoder
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from rank_bm25 import BM25Okapi
from google import genai
from google.colab import userdata
from google.colab import drive

/tmp/ipykernel_751/3181315876.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:


# ── Mount Google Drive ────────────────────────────────────────────────────
print("🔄 Requesting Google Drive access mount...")
drive.mount('/content/drive')

# ── Permanent Storage Coordinates ─────────────────────────────────────────
BASE_DIR = Path("/content/drive/MyDrive/rbi_farmer_data")
DB_DIR   = Path("/content/drive/MyDrive/rbi_chroma_db_v3")

# Global Configuration Parameters
EMBED_MODEL  = "BAAI/bge-large-en-v1.5"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-12-v2"

PDF_CHUNK_SIZE    = 1600
PDF_CHUNK_OVERLAP = 350
FAQ_CHUNK_TARGET  = 1400

BM25_TOP_K   = 20
DENSE_TOP_K  = 20
FINAL_TOP_K  = 6

CATEGORIES = {
    "credit_loans": BASE_DIR / "1_credit_loans",
    "insurance": BASE_DIR / "2_insurance",
    "banking_access": BASE_DIR / "3_banking_access",
    "govt_schemes": BASE_DIR / "4_govt_schemes",
    "grievance": BASE_DIR / "5_grievance",
    "educational": BASE_DIR / "6_educational",
}

for folder in CATEGORIES.values():
    folder.mkdir(parents=True, exist_ok=True)

# Scraper Structural Settings
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.rbi.org.in/",
    "Accept": "application/pdf,text/html,*/*",
}

AGRI_KEYWORDS = ["kisan", "kcc", "crop loan", "crop insurance", "agricultural credit", "agri", "agriculture", "farm loan", "farmer", "rural credit", "rural bank", "priority sector", "nabard", "microfinance", "land development", "jan dhan", "financial inclusion", "business correspondent", "interest subvention", "debt waiver", "loan waiver", "pm-kisan", "pmkisan", "pmfby", "enam", "soil health", "kisan vikas", "rupay kisan", "mudra", "self help group", "shg"]
JUNK_KEYWORDS = ["accessibility", "utkarsh", "judgment", "judgement", "court order", "election", "monetary policy", "basel", "forex", "foreign exchange", "nri deposit", "ecb guideline", "derivative", "hedge fund", "capital market", "securities"]

LOG_FILE = BASE_DIR / "download_log.json"
log = []
MAX_FILES_PER_CATEGORY = 40

print(f"\n✅ Persistent environment directories configured successfully.")

🔄 Requesting Google Drive access mount...
Mounted at /content/drive

✅ Persistent environment directories configured successfully.


In [ ]:
def over_limit(folder: Path) -> bool:
    count = len(list(folder.glob("*")))
    if count >= MAX_FILES_PER_CATEGORY:
        print(f"  ⚠️ Cap reached ({MAX_FILES_PER_CATEGORY}) for {folder.name} — skipping extraction.")
        return True
    return False

def is_relevant(text: str) -> bool:
    text = text.lower()
    if any(j in text for j in JUNK_KEYWORDS): return False
    return any(k in text for k in AGRI_KEYWORDS)

def make_filename(url: str, link_text: str = "") -> str:
    if link_text and len(link_text) > 5:
        name = re.sub(r'[<>:"/\\\\|?*\n\r\t]', "_", link_text.strip())
        name = re.sub(r'\s+', "_", name)[:80]
    else:
        name = url.split("/")[-1].split("?")[0]
        name = re.sub(r'[<>:"/\\\\|?*]', "_", name)
    if not name.lower().endswith(".pdf"): name += ".pdf"
    return name

def download_pdf(url: str, dest_folder: Path, filename: str = None, source: str = "") -> bool:
    if over_limit(dest_folder): return False
    try:
        r = requests.get(url, headers=HEADERS, timeout=30, stream=True)
        if r.status_code == 418:
            print(f"  ✗ 418 (blocked) — {url}")
            return False
        if r.status_code != 200:
            print(f"  ✗ {r.status_code} — {url}")
            return False
        content_type = r.headers.get("Content-Type", "")
        if "pdf" not in content_type.lower() and not url.lower().endswith(".pdf"): return False

        if not filename: filename = make_filename(url)
        dest = dest_folder / filename
        if dest.exists(): return True

        with open(dest, "wb") as f:
            for chunk in r.iter_content(8192): f.write(chunk)

        size_kb = dest.stat().st_size // 1024
        if size_kb < 5:
            dest.unlink() # Delete corrupt tiny files
            return False

        print(f"  ✓ {filename}  ({size_kb} KB)")
        log.append({"file": str(dest), "url": url, "source": source, "size_kb": size_kb})
        return True
    except Exception as e:
        print(f"  ✗ Error downloading {url}: {e}")
        return False

# ── STREAM 1: Curated Direct Manual Entries ───────────────────────────────
def download_curated_pdfs():
    print("\n📥 SCRAPER 1 — Pulling Curated Core PDFs...")
    CURATED = [
        {"url": "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/MDPSL032023B9BCA4F1B9C54BF3BE1D91F48E6D4E4F.PDF", "name": "master_direction_priority_sector_lending_2023.pdf", "cat": "credit_loans"},
        {"url": "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/KISANCC130621.PDF", "name": "kisan_credit_card_revised_guidelines.pdf", "cat": "credit_loans"},
        {"url": "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR48823.PDF", "name": "interest_subvention_scheme_farmers.pdf", "cat": "credit_loans"},
        {"url": "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/AGRILOAN1206.pdf", "name": "agri_loan_npa_rescheduling_guidelines.pdf", "cat": "credit_loans"},
        {"url": "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/MDBCMODEL020922.PDF", "name": "business_correspondent_model_master_direction.pdf", "cat": "banking_access"},
        {"url": "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/FINDEX2022.PDF", "name": "financial_inclusion_index_report_2022.pdf", "cat": "banking_access"},
        {"url": "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/BSBC130110.pdf", "name": "basic_savings_bank_account_jan_dhan.pdf", "cat": "banking_access"},
        {"url": "https://rbidocs.rbi.org.in/rdocs/content/pdfs/OMBUDSMAN121121.pdf", "name": "rbi_integrated_ombudsman_scheme_2021.pdf", "cat": "grievance"},
        {"url": "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/FPCNBFC010921.PDF", "name": "fair_practices_code_nbfc_lenders.pdf", "cat": "grievance"},
        {"url": "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/FPCSCB010921.PDF", "name": "fair_practices_code_scheduled_banks.pdf", "cat": "grievance"},
        {"url": "https://www.nabard.org/auth/writereaddata/tender/1407180417Financial%20Literacy%20for%20Farmers.pdf", "name": "nabard_financial_literacy_for_farmers.pdf", "cat": "educational"}
    ]
    for item in tqdm(CURATED, desc="Curated Pipeline"):
        download_pdf(item["url"], CATEGORIES[item["cat"]], filename=item["name"], source="Curated")
        time.sleep(1.2)

# ── STREAM 2: RBI FAQ Extraction Track ────────────────────────────────────
def scrape_rbi_faqs():
    print("\n📥 SCRAPER 2 — Parsing Structural RBI FAQ Panels...")
    FAQ_URLS = {
        "faq_kisan_credit_card": ("https://www.rbi.org.in/Scripts/FAQView.aspx?Id=79", "credit_loans"),
        "faq_priority_sector": ("https://www.rbi.org.in/Scripts/FAQView.aspx?Id=75", "credit_loans"),
        "faq_financial_inclusion": ("https://www.rbi.org.in/Scripts/FAQView.aspx?Id=54", "banking_access"),
        "faq_banking_ombudsman": ("https://www.rbi.org.in/Scripts/FAQView.aspx?Id=30", "grievance"),
        "faq_jan_dhan": ("https://www.rbi.org.in/Scripts/FAQView.aspx?Id=108", "banking_access"),
        "faq_rural_credit": ("https://www.rbi.org.in/Scripts/FAQView.aspx?Id=71", "credit_loans"),
        "faq_microfinance": ("https://www.rbi.org.in/Scripts/FAQView.aspx?Id=83", "credit_loans"),
        "faq_recovery_agents": ("https://www.rbi.org.in/Scripts/FAQView.aspx?Id=49", "grievance"),
    }
    for name, (url, cat) in FAQ_URLS.items():
        try:
            dest = CATEGORIES[cat] / f"{name}.txt"
            if dest.exists(): continue
            r = requests.get(url, headers=HEADERS, timeout=20)
            soup = BeautifulSoup(r.text, "html.parser")
            content = soup.find("div", {"id": "accordion"}) or soup.find("div", class_="tablebg") or soup.find("div", class_="content")
            text = content.get_text("\n", strip=True) if content else soup.get_text("\n", strip=True)
            text = re.sub(r'\n{3,}', '\n\n', text)
            dest.write_text(text, encoding="utf-8")
            print(f"  ✓ Captured FAQ Content: {name}.txt ({len(text)} chars)")
            log.append({"file": str(dest), "url": url, "source": "RBI FAQ", "chars": len(text)})
            time.sleep(1)
        except Exception as e: print(f"  ❌ FAQ Extraction Error {name}: {e}")

# ── STREAM 3: Master Directions Aggregator ────────────────────────────────
def scrape_rbi_master_directions():
    print("\n📥 SCRAPER 3 — Crawling RBI Master Directions Repository...")
    INDEX_URL = "https://www.rbi.org.in/Scripts/BS_ViewMasDirections.aspx"
    try:
        r = requests.get(INDEX_URL, headers=HEADERS, timeout=20)
        soup = BeautifulSoup(r.text, "html.parser")
        found = 0
        for a in soup.find_all("a", href=True):
            link_text = a.get_text(strip=True)
            if not is_relevant(link_text): continue
            full_url = urljoin(INDEX_URL, a["href"])
            if ".pdf" in full_url.lower():
                fname = make_filename(full_url, link_text)
                if download_pdf(full_url, CATEGORIES["credit_loans"], filename=fname, source="RBI Master Directions"): found += 1
            else:
                try:
                    detail = requests.get(full_url, headers=HEADERS, timeout=15)
                    dsoup = BeautifulSoup(detail.text, "html.parser")
                    for pdf_a in dsoup.find_all("a", href=re.compile(r'\.pdf', re.I)):
                        pdf_url = urljoin(full_url, pdf_a["href"])
                        fname = make_filename(pdf_url, link_text)
                        if download_pdf(pdf_url, CATEGORIES["credit_loans"], filename=fname, source="RBI Master Directions"): found += 1
                    time.sleep(0.5)
                except: pass
            time.sleep(0.8)
        print(f"  → Collected {found} matching Master Direction files.")
    except Exception as e: print(f"  ❌ Master Directions Extraction Interrupted: {e}")

# ── STREAM 4: Dynamic Notifications Filter ────────────────────────────────
def scrape_rbi_notifications():
    print("\n📥 SCRAPER 4 — Crawling Live RBI Notifications Index...")
    NOTIF_URL = "https://www.rbi.org.in/Scripts/NotificationUser.aspx"
    try:
        r = requests.get(NOTIF_URL, headers=HEADERS, timeout=20)
        soup = BeautifulSoup(r.text, "html.parser")
        found = 0
        for a in soup.find_all("a", href=True):
            link_text = a.get_text(strip=True)
            if not is_relevant(link_text): continue
            full_url = urljoin(NOTIF_URL, a["href"])
            if ".pdf" in full_url.lower():
                fname = make_filename(full_url, link_text)
                if download_pdf(full_url, CATEGORIES["credit_loans"], filename=fname, source="RBI Notifications"): found += 1
                time.sleep(0.8)
        print(f"  → Collected {found} relevant matching notification profiles.")
    except Exception as e: print(f"  ❌ Notifications Stream Failure: {e}")

# ── STREAM 5: NABARD Repository Crawl ─────────────────────────────────────
def scrape_nabard():
    print("\n📥 SCRAPER 5 — Crawling NABARD Publication Hub...")
    NABARD_PAGES = [
        ("https://www.nabard.org/content.aspx?id=572", "educational"),
        ("https://www.nabard.org/content.aspx?id=583", "credit_loans"),
        ("https://www.nabard.org/content.aspx?id=580", "credit_loans")
    ]
    for page_url, cat in NABARD_PAGES:
        try:
            r = requests.get(page_url, headers=HEADERS, timeout=20)
            soup = BeautifulSoup(r.text, "html.parser")
            for a in soup.find_all("a", href=re.compile(r'\.pdf', re.I)):
                link_text = a.get_text(strip=True)
                pdf_url = urljoin(page_url, a["href"])
                fname = make_filename(pdf_url, link_text)
                download_pdf(pdf_url, CATEGORIES[cat], filename=fname, source="NABARD")
                time.sleep(0.8)
        except Exception as e: print(f"  ❌ NABARD Connection Blocked on {page_url}: {e}")

# ── STREAM 6: Financial Literacy Database ─────────────────────────────────
def scrape_financial_literacy():
    print("\n📥 SCRAPER 6 — Gathering RBI Financial Literacy Materials...")
    PAGES = [
        "https://www.rbi.org.in/FinancialEducation/",
        "https://www.rbi.org.in/Scripts/PublicationsView.aspx?id=20882"
    ]
    for page_url in PAGES:
        try:
            r = requests.get(page_url, headers=HEADERS, timeout=20)
            soup = BeautifulSoup(r.text, "html.parser")
            for a in soup.find_all("a", href=re.compile(r'\.pdf', re.I)):
                link_text = a.get_text(strip=True)
                if not is_relevant(link_text) and not any(k in link_text.lower() for k in ["financial", "literacy", "education"]): continue
                pdf_url = urljoin(page_url, a["href"])
                fname = make_filename(pdf_url, link_text)
                download_pdf(pdf_url, CATEGORIES["educational"], filename=fname, source="RBI Financial Literacy")
                time.sleep(0.8)
        except Exception as e: print(f"  ❌ Financial Literacy Pull Faulted: {e}")

def run_orchestrated_scraper():
    total_existing_files = sum(1 for folder in CATEGORIES.values() for f in folder.glob("*") if f.is_file())
    if total_existing_files > 5:
        print(f"📦 Drive Verification Complete: Found {total_existing_files} active files inside storage. Scraper sync skipped.")
    else:
        print("🚀 Local Drive cache unverified. Triggering multi-stream scraper infrastructure...")
        download_curated_pdfs()
        scrape_rbi_faqs()
        scrape_rbi_master_directions()
        scrape_rbi_notifications()
        scrape_nabard()
        scrape_financial_literacy()

        # Write out execution manifest records
        with open(LOG_FILE, "w") as f: json.dump(log, f, indent=2)

run_orchestrated_scraper()

📦 Drive Verification Complete: Found 43 active files inside storage. Scraper sync skipped.


In [ ]:
def clean_text(text: str) -> str:
    if not text: return ""
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    text = re.sub(r'\r\n|\r|\f', '\n', text)
    text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'www\.rbi\.org\.in', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Reserve Bank of India\s*[-–]\s*[A-Za-z ]+', '', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    lines = [line for line in text.split('\n') if not re.match(r'^\d{1,4}$', line.strip())]
    text = '\n'.join(lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return re.sub(r'[^\x09\x0A\x0D\x20-\x7E\u0900-\u097F\u20B9]', ' ', text).strip()

def add_chunk_header(text: str, source: str, category: str) -> str:
    cat_display = re.sub(r'^\d+\s+', '', category.replace('_', ' ')).title()
    return f"[Document: {source} | Category: {cat_display}]\n{text}"

SECTION_RE = re.compile(
    r'\n(?=\d+\.\d*(?:\.\d+)?\s+[A-Z]|\b(?:PART|CHAPTER|ANNEX|SCHEDULE|PARAGRAPH|SECTION|APPENDIX)\s+[A-Z0-9I]|\b[A-Z][A-Z ]{4,}:\s)',
    re.MULTILINE
)
FALLBACK_SPLITTER = RecursiveCharacterTextSplitter(chunk_size=PDF_CHUNK_SIZE, chunk_overlap=PDF_CHUNK_OVERLAP, separators=["\n\n", "\n", ". ", " "])

def load_pdf(file_path: str) -> str:
    try:
        return "\n\n".join(p.page_content for p in PyPDFLoader(file_path).load() if len(p.page_content.strip()) > 300)
    except Exception: pass
    try:
        with pdfplumber.open(file_path) as pdf:
            return "\n\n".join(page.extract_text(x_tolerance=2, y_tolerance=2) or "" for page in pdf.pages)
    except Exception: return ""

def chunk_pdf(text: str, filename: str, category: str) -> list[Document]:
    sections = SECTION_RE.split(text)
    docs, prev_tail = [], ""
    for section in sections:
        section = section.strip()
        if not section: continue
        combined = f"{prev_tail}\n\n{section}".strip() if prev_tail else section
        if len(combined) <= PDF_CHUNK_SIZE:
            docs.append(Document(page_content=add_chunk_header(combined, filename, category), metadata={"source": filename, "category": category, "strategy": "section_aware"}))
        else:
            sub = FALLBACK_SPLITTER.create_documents([combined])
            for d in sub:
                d.page_content = add_chunk_header(d.page_content, filename, category)
                d.metadata = {"source": filename, "category": category, "strategy": "section_fallback"}
            docs.extend(sub)
        prev_tail = section[-PDF_CHUNK_OVERLAP:] if len(section) > PDF_CHUNK_OVERLAP else section
    return docs

def chunk_faq(text: str, filename: str, category: str) -> list[Document]:
    blocks = re.split(r'\n(?=\d+\.|\bQ\d*[:.\s]|\bAns\b[:.\s])', text)
    docs, buffer = [], ""
    for block in blocks:
        block = block.strip()
        if not block: continue
        if len(buffer) + len(block) < FAQ_CHUNK_TARGET:
            buffer = f"{buffer}\n\n{block}".strip()
        else:
            if buffer: docs.append(Document(page_content=add_chunk_header(buffer, filename, category), metadata={"source": filename, "category": category, "strategy": "faq"}))
            buffer = block
    if buffer: docs.append(Document(page_content=add_chunk_header(buffer, filename, category), metadata={"source": filename, "category": category, "strategy": "faq"}))
    return docs

def build_chunks(data_dir: Path) -> list[Document]:
    all_files = [f for f in data_dir.glob("**/*") if f.is_file() and f.name != "download_log.json"]
    all_chunks = []
    print(f"✂️ Analyzing structure tracks inside structural folders: {data_dir}...")
    for fp in tqdm(all_files, desc="Parsing Workspace Elements"):
        category, suffix = fp.parent.name, fp.suffix.lower()
        if suffix == '.txt':
            text = clean_text(fp.read_text(encoding='utf-8', errors='ignore'))
            if len(text) >= 80: all_chunks.extend(chunk_faq(text, fp.name, category))
        elif suffix == '.pdf':
            text = clean_text(load_pdf(str(fp)))
            if text.strip(): all_chunks.extend(chunk_pdf(text, fp.name, category))
    print(f"\n🎉 Structured extraction complete. Generated {len(all_chunks)} target context chunks.")
    return all_chunks

# ── FIX 1: Actually call build_chunks so `chunks` exists for Cell 5 ────────
chunks = build_chunks(BASE_DIR)

# ── FIX 2: tokenise defined here so it's available for BM25 build below
#           (previously only defined in Cell 6 — too late) ─────────────────
def tokenise(text: str) -> list[str]:
    return re.findall(r'[a-z0-9][a-z0-9.%-]*', text.lower())

# ── FIX 3: Build and persist BM25 index to Drive so Cell 6 can load it ────
print("🔍 Building BM25 sparse index...")
corpus_texts = [doc.page_content for doc in chunks]
corpus_meta  = [doc.metadata for doc in chunks]
bm25_index   = BM25Okapi([tokenise(t) for t in corpus_texts])

bm25_save_path = DB_DIR / "bm25_index.pkl"
DB_DIR.mkdir(parents=True, exist_ok=True)
with open(bm25_save_path, "wb") as f:
    pickle.dump((bm25_index, corpus_texts, corpus_meta), f)
print(f"✅ BM25 index saved → {bm25_save_path}")

✂️ Analyzing structure tracks inside structural folders: /content/drive/MyDrive/rbi_farmer_data...


Parsing Workspace Elements: 100%|██████████| 43/43 [03:21<00:00,  4.68s/it]



🎉 Structured extraction complete. Generated 7320 target context chunks.
🔍 Building BM25 sparse index...
✅ BM25 index saved → /content/drive/MyDrive/rbi_chroma_db_v3/bm25_index.pkl


In [ ]:
import os
import shutil
import chromadb
from chromadb.config import Settings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from pathlib import Path

# ── Constants ──────────────────────────────────────────────────────────────
EMBED_MODEL        = "BAAI/bge-large-en-v1.5"
DB_DIR             = Path("/content/drive/MyDrive/rbi_chroma_db_v3")  # Drive (final storage)
LOCAL_DB_PATH      = "/content/rbi_chroma_db_v3"                      # Local disk (build here)
COLLECTION_NAME    = "rbi_farmers_collection_v3"

# ── Step 1: Build on local disk (Drive FUSE can't handle SQLite WAL mode) ─
if os.path.exists(LOCAL_DB_PATH):
    shutil.rmtree(LOCAL_DB_PATH, ignore_errors=True)
os.makedirs(LOCAL_DB_PATH, exist_ok=True)
print(f"✅ Local build directory ready: {LOCAL_DB_PATH}")

# ── Step 2: Clear in-process chromadb singleton ────────────────────────────
chromadb.api.ClientAPI.clear_system_cache()

# ── Step 3: Initialize embedder ────────────────────────────────────────────
print("🧬 Initializing Embeddings...")
embedder = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

# ── Step 4: Create ChromaDB client on LOCAL disk ───────────────────────────
print(f"💾 Creating Fresh Chroma Vector Store with {len(chunks)} chunks...")
client = chromadb.PersistentClient(
    path=LOCAL_DB_PATH,
    settings=Settings(anonymized_telemetry=False, allow_reset=True)
)

try:
    client.delete_collection(name=COLLECTION_NAME)
except:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)
print(f"✅ Created collection: {COLLECTION_NAME}")

vector_db = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedder,
)

# ── Step 5: Insert in batches of 5000 (ChromaDB max is 5461) ──────────────
BATCH_SIZE = 5000
total = len(chunks)
print("📥 Adding documents in batches...")

for i in range(0, total, BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    vector_db.add_documents(documents=batch)
    print(f"  ✅ Batch {i//BATCH_SIZE + 1}: chunks {i+1}–{min(i+BATCH_SIZE, total)} of {total}")

print(f"\n✅ Successfully added {vector_db._collection.count()} documents!")

# ── Step 6: Copy finished DB from local disk → Google Drive ───────────────
print(f"\n📦 Copying DB to Google Drive...")
if os.path.exists(str(DB_DIR)):
    shutil.rmtree(str(DB_DIR), ignore_errors=True)
shutil.copytree(LOCAL_DB_PATH, str(DB_DIR))
print(f"✅ DB saved to Drive: {DB_DIR}")

✅ Local build directory ready: /content/rbi_chroma_db_v3
🧬 Initializing Embeddings...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


💾 Creating Fresh Chroma Vector Store with 7320 chunks...
✅ Created collection: rbi_farmers_collection_v3
📥 Adding documents in batches...
  ✅ Batch 1: chunks 1–5000 of 7320
  ✅ Batch 2: chunks 5001–7320 of 7320

✅ Successfully added 7320 documents!

📦 Copying DB to Google Drive...
✅ DB saved to Drive: /content/drive/MyDrive/rbi_chroma_db_v3


In [ ]:
import chromadb
chromadb.api.ClientAPI.clear_system_cache()
print("Cache cleared! You can now safely initialize a new client instance.")


Cache cleared! You can now safely initialize a new client instance.


In [ ]:
import os
import shutil
import pickle
import re
import time  # <--- Added for performance tracking
import chromadb
from chromadb.config import Settings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
from google import genai
from google.colab import userdata

# ── 0. Move DB Local for Fast SQLite Access ────────────────────────────────
LOCAL_DB_PATH = "/content/rbi_chroma_db_v3"
if not os.path.exists(LOCAL_DB_PATH):
    print("🚀 Copying database to local high-speed disk...")
    shutil.copytree(str(DB_DIR), LOCAL_DB_PATH)

# ── 1. Clear Local Cache Prior to Initialization ───────────────────────────
chromadb.api.ClientAPI.clear_system_cache()

# ── 2. Initialize Embedder Module (Now using CUDA!) ────────────────────────
embedder = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

# ── 3. Connect to Local Vector DB via Explicit Client ──────────────────────
persistent_client = chromadb.PersistentClient(
    path=LOCAL_DB_PATH,
    settings=Settings(anonymized_telemetry=False, allow_reset=True)
)

vector_db = Chroma(
    client=persistent_client,
    collection_name="rbi_farmers_collection_v3",
    embedding_function=embedder,
)

# ── 4. Load BM25 Index and Corpus Assets ──────────────────────────────────
with open(f"{DB_DIR}/bm25_index.pkl", "rb") as f:
    bm25_index, corpus_texts, corpus_meta = pickle.load(f)

print("🔀 Instantiating Neural Cross-Encoder Re-Ranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2", max_length=512, device="cuda")

print("🤖 Connecting system runtime parameters directly to Gemini Client Core...")
gemini_client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY"),
    http_options={"api_version": "v1beta"}
)
GEMINI_MODEL = "gemini-2.5-flash"

# ── 5. Operational Core Infrastructure functions ──────────────────────────
def tokenise(text: str) -> list[str]:
    return re.findall(r'[a-z0-9][a-z0-9.%-]*', text.lower())

def rrf(rankings: list[list[int]], k: int = 60) -> list[int]:
    scores = {}
    for r_list in rankings:
        for rank, idx in enumerate(r_list):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=lambda x: -scores[x])

def retrieve(query: str, verbose: bool = True) -> list[dict]:
    t0 = time.time()

    bm25_scores = bm25_index.get_scores(tokenise(query))
    bm25_ranked = sorted(range(len(corpus_texts)), key=lambda i: -bm25_scores[i])[:BM25_TOP_K]
    # ──────────────────────────────────────────────────────────────────

    t1 = time.time()
    if verbose: print(f"   ⚡ BM25 Keyword Search: {t1-t0:.2f} seconds")

    dense_ranked = []
    for dr in vector_db.similarity_search(query, k=DENSE_TOP_K):
        try:
            dense_ranked.append(corpus_texts.index(dr.page_content))
        except ValueError:
            pass

    t2 = time.time()
    if verbose: print(f"   ⚡ Dense Vector Search (GPU): {t2-t1:.2f} seconds")

    candidates = rrf([bm25_ranked, dense_ranked])[:30]
    rerank_scores = reranker.predict([(query, corpus_texts[i]) for i in candidates])
    top = sorted(zip(candidates, rerank_scores), key=lambda x: -x[1])[:FINAL_TOP_K]

    t3 = time.time()
    if verbose: print(f"   ⚡ Cross-Encoder Reranking (GPU): {t3-t2:.2f} seconds")

    return [
        {
            "text": corpus_texts[idx],
            "source": corpus_meta[idx].get("source", "unknown"),
            "category": corpus_meta[idx].get("category", "unknown"),
            "score": round(float(scr), 4)
        }
        for idx, scr in top
    ]

SYSTEM_PROMPT = """You are an expert regulatory assistant for Indian banking, agricultural credit schemes, and RBI directions.
CRITICAL OPERATIONAL RULES:
1. Ground your extraction ONLY on the Context Blocks structured below. Avoid tracking outside general knowledge base structures.
2. Provide explicit reference citations using inline markers stating [Source: <filename>].
3. If the context does not hold explicit answers, state clearly: "I couldn't find this in the available documents. Please check directly with your bank or the RBI website."
4. Be accurate with data: preserve explicit percentages, timelines, and multi-tier transaction cap limits.
5. Format your output cleanly in broken down, actionable bullet-points. Do not assume or guess."""

def ask(query: str, verbose: bool = True) -> str:
    print(f"🔄 Starting retrieval for: {query[:30]}...")
    results = retrieve(query, verbose=True) # Forced to true to see the speedometer

    if not results:
        return "I couldn't find this in the available documents. Please check directly with your bank or the RBI website."

    context = "\n\n".join([f"--- Context Block {i+1} [Source: {r['source']}] ---\n{r['text']}" for i, r in enumerate(results)])
    prompt = f"{SYSTEM_PROMPT}\n\n=== CONTEXT BLOCKS ===\n{context}\n=== END OF CONTEXT ===\n\nQUESTION: {query}\nANSWER:"

    print("🤖 Sending context to Gemini API...")
    t_start = time.time()

    response = gemini_client.models.generate_content(model=GEMINI_MODEL, contents=prompt).text.strip()

    t_end = time.time()
    print(f"   ⚡ Gemini Generation: {t_end-t_start:.2f} seconds")

    return response

print("✅ Hybrid multi-stream retrieval framework successfully initialized on GPU.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔀 Instantiating Neural Cross-Encoder Re-Ranker...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🤖 Connecting system runtime parameters directly to Gemini Client Core...
✅ Hybrid multi-stream retrieval framework successfully initialized on GPU.


In [ ]:
TEST_QUESTIONS = [
    "What is the maximum loan limit available under the Kisan Credit Card (KCC) scheme without needing to give collateral security?",
    "What percentage of bank credit must go to the agriculture sector under Priority Sector Lending?",
    "What are the capital requirements and liquidity coverage ratios under Basel III for global systemically important banks?"
]

print("🚀 Running target operational verification queries...")
for q in TEST_QUESTIONS:
    print(f"\n🔍 Query Trace: '{q}'")
    print("-" * 75)
    print(f"💬 Chatbot Response:\n{ask(q, verbose=False)}")
    print("=" * 75)

🚀 Running target operational verification queries...

🔍 Query Trace: 'What is the maximum loan limit available under the Kisan Credit Card (KCC) scheme without needing to give collateral security?'
---------------------------------------------------------------------------
🔄 Starting retrieval for: What is the maximum loan limit...
   ⚡ BM25 Keyword Search: 0.06 seconds
   ⚡ Dense Vector Search (GPU): 0.03 seconds
   ⚡ Cross-Encoder Reranking (GPU): 0.38 seconds
🤖 Sending context to Gemini API...
   ⚡ Gemini Generation: 2.31 seconds
💬 Chatbot Response:
Based on the provided documents:

*   Collateral security is waived for a loan limit of up to Rs. 1 lakh under the Kisan Credit Card (KCC) scheme. [Source: 03FARMERS20042018.pdf]

🔍 Query Trace: 'What percentage of bank credit must go to the agriculture sector under Priority Sector Lending?'
---------------------------------------------------------------------------
🔄 Starting retrieval for: What percentage of bank credit...
   ⚡ BM25 Ke

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
print("🌾 RBI Agriculture Assistant Online. Type 'exit' to terminate session.")
while True:
    try:
        user_input = input("\n❓ Farmer Query: ").strip()
    except (EOFError, KeyboardInterrupt): break
    if not user_input or user_input.lower() in {"exit", "quit"}: break
    print(f"\n🤖 Answer:\n{ask(user_input, verbose=False)}")

🌾 RBI Agriculture Assistant Online. Type 'exit' to terminate session.
🔄 Starting retrieval for: What are the loan limits and t...
   ⚡ BM25 Keyword Search: 0.07 seconds
   ⚡ Dense Vector Search (GPU): 0.04 seconds
   ⚡ Cross-Encoder Reranking (GPU): 0.43 seconds
🤖 Sending context to Gemini API...
   ⚡ Gemini Generation: 11.32 seconds

🤖 Answer:
Based on the provided documents, here are the details regarding the target groups and available information on loan limits for KCC under animal husbandry and fisheries:

*   **Target Groups:**
    *   Eligible farmers engaged in animal husbandry, dairying, and fisheries [Source: version-final.pdf].
    *   Individual farmers (including Self Help Groups (SHGs) or Joint Liability Groups (JLGs) of individual farmers, and Proprietorship firms of farmers) directly engaged in allied activities such as dairy, fishery, animal husbandry, poultry, bee-keeping, and sericulture are eligible for Farm Credit, which includes loans under the Kisan Credit Card S